# 22 — Supervised Fine-Tuning from First Principles

## Goal

Understand how a pretrained autoregressive language model
learns to follow instructions through supervised examples.

We focus on:

- Conversation formatting and role boundaries.
- Teacher-forced next-token prediction.
- Shifted labels and assistant-only loss masking.
- Fine-tuning a pretrained GPT.
- Evaluating instruction-following behavior.

## 01. SFT Objective and Label Alignment

The model still predicts the next token.

The difference is which tokens provide supervision.

For assistant-only SFT, the prompt remains part of the
conditioning context, but only selected assistant targets
contribute to the training loss.

In [1]:
import torch
import torch.nn.functional as F
from llmfp.models import GPT, GPTConfig

In [2]:
# Align next-token targets with an assistant-only loss mask.

# Vocabulary:
# 0 = BOS
# 1 = USER
# 2 = QUESTION
# 3 = ASSISTANT
# 4 = ANSWER
# 5 = EOS

# (B=1, L=6): sequences, full conversation token positions
token_ids = torch.tensor([[0, 1, 2, 3, 4, 5]])

# (B=1, T=5): context tokens supplied to the language model
input_ids = token_ids[:, :-1]

# (B=1, T=5): next-token labels for each input position
target_ids = token_ids[:, 1:]

# (B=1, T=5): which target positions contribute to SFT loss
assistant_mask = torch.tensor([[False, False, False, True, True]])

assert input_ids.shape == target_ids.shape == assistant_mask.shape
assert torch.equal(target_ids[assistant_mask], torch.tensor([4, 5]))

print("Inputs:", input_ids.tolist())
print("Targets:", target_ids.tolist())
print("Loss mask:", assistant_mask.tolist())

Inputs: [[0, 1, 2, 3, 4]]
Targets: [[1, 2, 3, 4, 5]]
Loss mask: [[False, False, False, True, True]]


In [3]:
# Verify that only assistant targets contribute gradients.

vocab_size = 6

# (B=1, T=5, V=6): next-token logits for every prediction position
logits = torch.zeros((1, 5, vocab_size), requires_grad=True)

# (B=1, T=5): unreduced next-token negative log-likelihoods
per_token_loss = F.cross_entropy(
    logits.reshape(-1, vocab_size), target_ids.reshape(-1), reduction="none"
).reshape(1, 5)

# (): mean NLL over assistant target positions only
sft_loss = per_token_loss[assistant_mask].mean()

sft_loss.backward()

assert logits.grad is not None

# Prompt prediction positions must contribute zero gradients.
assert torch.all(logits.grad[:, :3, :] == 0)

# Assistant prediction positions must receive gradients.
assert torch.any(logits.grad[:, 3:, :] != 0)

print("Masked SFT loss:", sft_loss.item())
print("Assistant-only gradient test passed.")

Masked SFT loss: 1.7917594909667969
Assistant-only gradient test passed.


## 02. Instruction Data and Chat Templates

A chat template converts structured role/content messages into
a token sequence understood by the language model.

For our first experiment, we define a minimal single-turn format:

[BOS, USER, question..., ASSISTANT, answer..., EOS]

The prompt and role markers provide context. We select the answer
tokens and EOS as supervised prediction targets.

This is a pedagogical symbolic vocabulary, not a pretrained
model tokenizer.

In [4]:
BOS_ID = 0
USER_ID = 1
ASSISTANT_ID = 3
EOS_ID = 5


def encode_sft_pair(
    question: list[int],
    answer: list[int],
) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    """Serialize one instruction-response pair for assistant-only SFT.

    Args:
        question: Token IDs of the user content.
        answer: Token IDs of the assistant response.

    Returns:
        input_ids: (T,) — context tokens.
        target_ids: (T,) — next-token labels.
        loss_mask: (T,) — True for answer and EOS targets.
    """
    if not question or not answer:
        raise ValueError("Question and answer must be non-empty.")

    # Serialize the full conversation, including role delimiters.
    full_tokens = [BOS_ID, USER_ID, *question, ASSISTANT_ID, *answer, EOS_ID]

    # (L,): complete conversation token positions
    sequence = torch.tensor(full_tokens, dtype=torch.long)

    # Position of the first assistant answer token in the full sequence
    answer_start = 3 + len(question)

    # (T=L-1,): input tokens for next-token prediction
    input_ids = sequence[:-1]

    # (T=L-1,): next-token targets, shifted by one position
    target_ids = sequence[1:]

    # (T=L-1,): supervised target positions
    loss_mask = torch.zeros_like(target_ids, dtype=torch.bool)

    # Target index is one position earlier than its full-sequence index.
    # Include answer tokens and EOS, but exclude the assistant role marker.
    loss_mask[answer_start - 1 :] = True

    return input_ids, target_ids, loss_mask

In [5]:
# Two user-content tokens and two assistant-content tokens
question = [2, 6]
answer = [4, 7]

input_ids, target_ids, loss_mask = encode_sft_pair(question, answer)

print("Inputs: ", input_ids.tolist())
print("Targets:", target_ids.tolist())
print("Mask:   ", loss_mask.tolist())

# Only answer tokens and EOS receive supervised loss.
torch.testing.assert_close(
    target_ids[loss_mask],
    torch.tensor([4, 7, 5]),
)

# Two answer tokens plus one end-of-sequence token
assert loss_mask.sum().item() == 3

print("SFT encoding test passed.")

Inputs:  [0, 1, 2, 6, 3, 4, 7]
Targets: [1, 2, 6, 3, 4, 7, 5]
Mask:    [False, False, False, False, True, True, True]
SFT encoding test passed.


### Chat Templates and SFT Encoding — Takeaways

A chat template maps structured messages into a deterministic
token sequence containing role delimiters and content tokens.

For a single-turn assistant-only SFT example, we constructed:

- Shifted model inputs.
- Next-token prediction targets.
- A Boolean mask selecting assistant answer tokens and EOS.

The mask is aligned with target positions, not the original
unshifted conversation indices.

Causal attention masks and supervision masks solve different
problems: one controls accessible information; the other controls
which prediction errors contribute to optimization.

The encoding is intentionally minimal. Multi-turn conversations,
real tokenizers, padding, and truncation require additional care.

In [ ]:
torch.manual_seed(42)

sft_config = GPTConfig(
    vocab_size=8,
    embedding_dim=32,
    num_query_heads=4,
    num_kv_heads=2,
    num_layers=2,
    hidden_dim=64,
)

sft_model = GPT(sft_config)
sft_model.train()

# Each tensor has shape (T=7,): shifted token positions.
input_ids, target_ids, loss_mask = encode_sft_pair(
    question=[2, 6], answer=[4, 7]
)

# (B=1, T=7): sequences, input-token positions
sft_inputs = input_ids.unsqueeze(0)

# (B=1, T=7): sequences, next-token targets
sft_targets = target_ids.unsqueeze(0)

# (B=1, T=7): supervised assistant-target positions
sft_mask = loss_mask.unsqueeze(0)

assert sft_inputs.shape == sft_targets.shape == sft_mask.shape
assert sft_mask.sum().item() == 3

print("Inputs:", sft_inputs.tolist())
print("Targets:", sft_targets.tolist())
print("Mask:", sft_mask.tolist())

Inputs: [[0, 1, 2, 6, 3, 4, 7]]
Targets: [[1, 2, 6, 3, 4, 7, 5]]
Mask: [[False, False, False, False, True, True, True]]


In [ ]:
# (B=1, T=7, V=8): next-token logits at every sequence position
sft_logits = sft_model(sft_inputs)

assert isinstance(sft_logits, torch.Tensor)

# (B=1, T=7): unreduced next-token negative log-likelihoods
per_token_loss = F.cross_entropy(
    sft_logits.reshape(-1, sft_config.vocab_size),
    sft_targets.reshape(-1),
    reduction="none",  # !not "mean"!
).reshape(sft_targets.shape)

# (): mean NLL over assistant targets only
sft_loss = per_token_loss[sft_mask].mean()

# (): ordinary next-token NLL over every position, for comparison
full_lm_loss = per_token_loss.mean()

print("Full LM loss:", full_lm_loss.item())
print("Assistant-only SFT loss:", sft_loss.item())

assert sft_loss.requires_grad
assert torch.isfinite(sft_loss)

Full LM loss: 2.136887311935425
Assistant-only SFT loss: 2.085982084274292


In [8]:
# Keep the graph because we will also backpropagate to model parameters.
# (B=1, T=7, V=8): gradients of SFT loss w.r.t. vocabulary logits
(logits_grad,) = torch.autograd.grad(
    outputs=sft_loss, inputs=(sft_logits,), retain_graph=True
)

# (N_prompt=4, V=8): gradients at masked predictor positions
prompt_logit_grads = logits_grad[~sft_mask]

# (N_assistant=3, V=8): gradients at supervised predictor positions
assistant_logit_grads = logits_grad[sft_mask]

assert torch.all(prompt_logit_grads == 0)
assert torch.any(assistant_logit_grads != 0)

print("Prompt logits gradient norm:", prompt_logit_grads.norm().item())

print("Assistant logits gradient norm:", assistant_logit_grads.norm().item())

print("Direct logit gradient masking passed.")

Prompt logits gradient norm: 0.0
Assistant logits gradient norm: 0.5508262515068054
Direct logit gradient masking passed.


In [9]:
sft_model.zero_grad(set_to_none=True)

sft_loss.backward()

# (V=8, C=32): gradients of the trainable token embedding table
embedding_grads = sft_model.token_embedding.weight.grad

assert embedding_grads is not None

# (N_prompt=5,): IDs appearing only in the prompt/context
prompt_token_ids = torch.tensor([BOS_ID, USER_ID, 2, 6, ASSISTANT_ID])

# (N_prompt=5, C=32): embedding gradients for prompt-only tokens
prompt_embedding_grads = embedding_grads[prompt_token_ids]

# (N_prompt=5,): gradient norm of each prompt token embedding
prompt_embedding_norms = prompt_embedding_grads.norm(dim=-1)

print("Prompt token IDs:", prompt_token_ids.tolist())
print("Prompt embedding gradient norms:", prompt_embedding_norms.tolist())

assert torch.any(prompt_embedding_norms > 0)

print("Indirect prompt gradient flow passed.")

Prompt token IDs: [0, 1, 2, 6, 3]
Prompt embedding gradient norms: [0.03976085036993027, 0.03468206897377968, 0.02956317365169525, 0.03257538005709648, 0.2189846783876419]
Indirect prompt gradient flow passed.


In [11]:
# (C=32,): EOS input-embedding gradient
eos_embedding_grad = embedding_grads[EOS_ID]

# (V=8, C=32): gradients of the vocabulary output projection
lm_head_grads = sft_model.lm_head.weight.grad

assert lm_head_grads is not None

# (C=32,): EOS output-class projection gradient
eos_output_grad = lm_head_grads[EOS_ID]

print("EOS embedding gradient:", eos_embedding_grad.norm().item())
print("EOS LM-head gradient:", eos_output_grad.norm().item())

assert torch.all(eos_embedding_grad == 0)
assert eos_output_grad.norm().item() > 0

print("Input embedding / output head gradient distinction passed.")

EOS embedding gradient: 0.0
EOS LM-head gradient: 1.5281026363372803
Input embedding / output head gradient distinction passed.


### Assistant-only Loss and Gradient Flow — Takeaways

We implemented assistant-only SFT using a real causal GPT.

The objective selects assistant response targets from
teacher-forced next-token predictions.

We verified two distinct gradient properties:

1. Predictor logits at masked prompt positions receive no
   direct gradient from the SFT loss.
2. Prompt token embeddings can still receive gradients because
   supervised assistant predictions depend on earlier context
   through causal attention.

Loss masking controls which predictions are supervised.
It does not remove the prompt from the Transformer's context.

We also distinguished input embeddings from vocabulary output
projections using the EOS token.

Our GPT uses separate embedding and LM-head parameters; the
behavior would differ under weight tying.

The next step is to combine examples of different lengths
using padding and valid assistant-target masks.


## 04. Padding, Batching, and Loss Masks

SFT datasets contain instruction-response examples of different lengths.

To combine them into a dense batch, we right-pad the shifted sequences
to a common length.

We distinguish:

- `valid_mask`: positions belonging to the original example.
- `loss_mask`: positions whose next-token targets are supervised.

For assistant-only SFT, the loss mask selects answer tokens and EOS.
Padding positions must not contribute to the loss.


In [12]:
# Reuse encode_sft_pair from Section 02.
examples = [
    encode_sft_pair(question=[2], answer=[4]),
    encode_sft_pair(question=[2, 6], answer=[4, 7]),
]

batch_size = len(examples)
max_length = max(x.numel() for x, _, _ in examples)

# (B=2, T_max=7): sequences, padded predictor inputs
batch_inputs = torch.full(
    (batch_size, max_length), fill_value=BOS_ID, dtype=torch.long
)

# (B=2, T_max=7): sequences, padded next-token targets
batch_targets = torch.full_like(batch_inputs, BOS_ID)

# (B=2, T_max=7): positions belonging to original examples
batch_valid_mask = torch.zeros_like(batch_inputs, dtype=torch.bool)

# (B=2, T_max=7): selected assistant-target positions
batch_loss_mask = torch.zeros_like(batch_inputs, dtype=torch.bool)

for row, (inputs, targets, mask) in enumerate(examples):
    length = inputs.numel()

    batch_inputs[row, :length] = inputs
    batch_targets[row, :length] = targets
    batch_valid_mask[row, :length] = True
    batch_loss_mask[row, :length] = mask

print("Inputs:", batch_inputs.tolist())
print("Targets:", batch_targets.tolist())
print("Valid mask:", batch_valid_mask.tolist())
print("Loss mask:", batch_loss_mask.tolist())

assert batch_inputs.shape == (2, 7)
assert batch_loss_mask.sum().item() == 5
assert not batch_loss_mask[~batch_valid_mask].any().item()
assert batch_inputs.max().item() < sft_config.vocab_size


Inputs: [[0, 1, 2, 3, 4, 0, 0], [0, 1, 2, 6, 3, 4, 7]]
Targets: [[1, 2, 3, 4, 5, 0, 0], [1, 2, 6, 3, 4, 7, 5]]
Valid mask: [[True, True, True, True, True, False, False], [True, True, True, True, True, True, True]]
Loss mask: [[False, False, False, True, True, False, False], [False, False, False, False, True, True, True]]


In [13]:
# (B=2, T=7, V=8): next-token vocabulary logits
batch_logits = sft_model(batch_inputs)

# (B=2, T=7): unreduced NLL for every target position
batch_nll = F.cross_entropy(
    batch_logits.reshape(-1, sft_config.vocab_size),
    batch_targets.reshape(-1),
    reduction="none",
).reshape(batch_targets.shape)

# Prevent an empty supervised set from producing NaN.
if not batch_loss_mask.any().item():
    raise ValueError("No supervised targets in the batch.")

# (): Boolean-mask mean over supervised tokens
boolean_loss = batch_nll[batch_loss_mask].mean()

# (B=2, T=7): -100 marks targets excluded from CE
ignored_targets = batch_targets.masked_fill(~batch_loss_mask, -100)

# (): Cross-entropy ignores -100 targets in its mean
ignore_index_loss = F.cross_entropy(
    batch_logits.reshape(-1, sft_config.vocab_size),
    ignored_targets.reshape(-1),
    ignore_index=-100,
)

torch.testing.assert_close(boolean_loss, ignore_index_loss)

print("Boolean-mask loss:", boolean_loss.item())
print("Ignore-index loss:", ignore_index_loss.item())
print("Supervised tokens:", batch_loss_mask.sum().item())


Boolean-mask loss: 1.987636923789978
Ignore-index loss: 1.987636923789978
Supervised tokens: 5


In [14]:
# (B=2, T=7, V=8): logit gradients using Boolean masking
(grad_boolean,) = torch.autograd.grad(
    boolean_loss, batch_logits, retain_graph=True
)

# (B=2, T=7, V=8): logit gradients using ignore_index
(grad_ignored,) = torch.autograd.grad(ignore_index_loss, batch_logits)

torch.testing.assert_close(grad_boolean, grad_ignored)

# Unsupervised positions must have zero direct logit gradients.
assert torch.all(grad_boolean[~batch_loss_mask] == 0)

print("Both masking methods produce equivalent gradients.")


Both masking methods produce equivalent gradients.



### Padding, Batching, and Loss Masks — Takeaways

We constructed a batch containing variable-length SFT examples.

Right padding aligns their tensor shapes while preserving their
original token positions.

The valid mask identifies original sequence positions, whereas
the loss mask selects assistant response targets and EOS.

Boolean masking and `ignore_index=-100` produce equivalent
cross-entropy losses and gradients when they select the same
targets and use the same normalization.

Loss averaging is a modeling choice: token-average and
example-average objectives give different weights to examples
with different response lengths.

The next step is to fine-tune a pretrained GPT using a compatible
tokenizer and instruction dataset.



## 05. Fine-Tuning a Pretrained GPT

### 05.1 Restoring the Pretrained Model and Tokenizer

Fine-tuning starts from pretrained parameters rather than a random
initialization.

We reuse the Tiny Shakespeare checkpoint and reconstruct the
character-level tokenizer from the same training corpus.

The tokenizer vocabulary and the model's input/output dimensions
must remain compatible.


In [15]:
from pathlib import Path
from llmfp.data import CharacterTokenizer

# Reuse the data and best-checkpoint paths from Lesson 16.
corpus_path = Path("../data/raw/tinyshakespeare.txt")
checkpoint_path = Path("../checkpoints/tiny_shakespeare/step_02000.pt")

for path in (corpus_path, checkpoint_path):
    if not path.is_file():
        raise FileNotFoundError(
            f"Missing {path}; working directory: {Path.cwd()}"
        )

corpus_text = corpus_path.read_text(encoding="utf-8")

# Reconstruct the deterministic character vocabulary.
pretrain_tokenizer = CharacterTokenizer.from_text(corpus_text)

assert pretrain_tokenizer.vocab_size == 65

print("Vocabulary size:", pretrain_tokenizer.vocab_size)
print("Checkpoint:", checkpoint_path.name)


Vocabulary size: 65
Checkpoint: step_02000.pt


In [16]:
# Lesson 16 saved a GPTConfig dataclass in the checkpoint.
with torch.serialization.safe_globals([GPTConfig]):
    pretrained_checkpoint = torch.load(
        checkpoint_path, map_location="cpu", weights_only=True
    )

pretrained_config = pretrained_checkpoint["model_config"]

assert isinstance(pretrained_config, GPTConfig)
assert pretrained_config.vocab_size == pretrain_tokenizer.vocab_size

# Restore the original architecture before loading the weights.
pretrained_gpt = GPT(pretrained_config)

pretrained_gpt.load_state_dict(
    pretrained_checkpoint["model_state_dict"],
    strict=True,
)

pretrained_gpt.eval()

print("Pretraining step:", pretrained_checkpoint["step"])
print("Vocabulary size:", pretrained_config.vocab_size)
print("Parameter count:", sum(p.numel() for p in pretrained_gpt.parameters()))


Pretraining step: 2000
Vocabulary size: 65
Parameter count: 3050240


In [17]:
prompt_text = "User: Say hello\nAssistant: "
answer_text = "Hello\n"

full_text = prompt_text + answer_text

prompt_tokens = pretrain_tokenizer.encode(prompt_text)
answer_tokens = pretrain_tokenizer.encode(answer_text)
full_tokens = pretrain_tokenizer.encode(full_text)

# (L,): complete character-token conversation
sequence = torch.tensor(full_tokens, dtype=torch.long)

# (B=1, T=L-1): shifted context tokens
pretrain_sft_inputs = sequence[:-1].unsqueeze(0)

# (B=1, T=L-1): shifted next-character targets
pretrain_sft_targets = sequence[1:].unsqueeze(0)

# (B=1, T=L-1): selected assistant-response target positions
pretrain_sft_mask = torch.zeros_like(pretrain_sft_targets, dtype=torch.bool)

# The first response character is predicted from the last prompt token.
answer_start = len(prompt_tokens) - 1
pretrain_sft_mask[:, answer_start:] = True

# (N_response,): answer characters and final newline targets
selected_targets = pretrain_sft_targets[pretrain_sft_mask]

torch.testing.assert_close(
    selected_targets,
    torch.tensor(answer_tokens),
)

print("Full text:", repr(full_text))
print(
    "Supervised text:",
    repr(pretrain_tokenizer.decode(selected_targets.tolist())),
)


Full text: 'User: Say hello\nAssistant: Hello\n'
Supervised text: 'Hello\n'


In [18]:
with torch.no_grad():
    # (B=1, T, V=65): pretrained model predictions
    baseline_logits = pretrained_gpt(pretrain_sft_inputs)

    # (B=1, T): negative log-likelihood for each target
    baseline_nll = F.cross_entropy(
        baseline_logits.reshape(-1, pretrained_config.vocab_size),
        pretrain_sft_targets.reshape(-1),
        reduction="none",
    ).reshape(pretrain_sft_targets.shape)

    # (): mean NLL over response characters and final newline
    baseline_sft_loss = baseline_nll[pretrain_sft_mask].mean()

print("Pre-fine-tuning SFT loss:", baseline_sft_loss.item())

assert torch.isfinite(baseline_sft_loss)


Pre-fine-tuning SFT loss: 3.8167755603790283



### Pretrained Model and Tokenizer — Takeaways

Fine-tuning begins from a pretrained checkpoint rather than
randomly initialized model parameters.

The tokenizer must preserve the original token-to-ID mapping.

Our Tiny Shakespeare GPT uses a character-level vocabulary with
65 tokens and has no dedicated chat-role or EOS tokens.

We serialized an instruction-response pair using ordinary
characters and aligned the assistant-only loss mask with
shifted targets.

We recorded a pre-fine-tuning SFT loss as a baseline.
No model parameters were updated in this step.



### 05.2 First Real SFT Updates

We will fine-tune the pretrained GPT on a small instruction-response
dataset using teacher forcing and assistant-only cross-entropy.

We will:

1. Encode instruction-response examples with the pretrained tokenizer.
2. Right-pad them into one batch.
3. Measure the loss before fine-tuning.
4. Update a copy of the pretrained model using AdamW.
5. Verify that the model parameters change and the training loss decreases.


In [19]:
training_pairs = [
    ("Say hello", "Hello"),
    ("Say farewell", "Farewell"),
    ("Say thanks", "Thanks"),
    ("Say yes", "Yes"),
]


def encode_character_sft_pair(
    instruction: str,
    response: str,
    tokenizer: CharacterTokenizer,
) -> tuple[torch.Tensor, torch.Tensor]:
    """Encode a text instruction-response pair for assistant-only SFT.

    Args:
        instruction: User instruction text.
        response: Expected assistant response.
        tokenizer: Character tokenizer from pretraining.

    Returns:
        input_ids: (T,) shifted predictor inputs.
        labels: (T,) shifted targets with prompt labels set to -100.
    """
    prompt = f"User: {instruction}\nAssistant: "
    answer = response + "\n"

    # (L,): full serialized conversation character positions
    sequence = torch.tensor(tokenizer.encode(prompt + answer), dtype=torch.long)

    # (T=L-1,): model input positions
    input_ids = sequence[:-1]

    # (T=L-1,): shifted target positions
    labels = sequence[1:].clone()

    # The first answer character is predicted at prompt_length - 1.
    prompt_length = len(tokenizer.encode(prompt))
    labels[: prompt_length - 1] = -100

    return input_ids, labels


encoded_examples = [
    encode_character_sft_pair(
        instruction,
        response,
        pretrain_tokenizer,
    )
    for instruction, response in training_pairs
]

for (instruction, response), (inputs, labels) in zip(
    training_pairs, encoded_examples
):
    print(
        instruction,
        "→",
        response,
        "| supervised tokens:",
        labels.ne(-100).sum().item(),
    )


Say hello → Hello | supervised tokens: 6
Say farewell → Farewell | supervised tokens: 9
Say thanks → Thanks | supervised tokens: 7
Say yes → Yes | supervised tokens: 4


In [20]:
batch_size = len(encoded_examples)
max_length = max(x.numel() for x, _ in encoded_examples)

# Use a valid space character ID for padded model inputs.
pad_input_id = pretrain_tokenizer.token_to_id[" "]

# (B=4, T_max=38): padded predictor input positions
train_inputs = torch.full(
    (batch_size, max_length), pad_input_id, dtype=torch.long
)

# (B=4, T_max=38): targets, ignoring prompt and padding
train_labels = torch.full((batch_size, max_length), -100, dtype=torch.long)

for row, (inputs, labels) in enumerate(encoded_examples):
    length = inputs.numel()

    train_inputs[row, :length] = inputs
    train_labels[row, :length] = labels

# (B=4, T_max=38): positions supervised by the SFT objective
supervised_mask = train_labels.ne(-100)

assert train_inputs.shape == train_labels.shape == (4, 38)
assert supervised_mask.sum().item() == 26
assert train_inputs.min().item() >= 0
assert train_inputs.max().item() < pretrained_config.vocab_size

print("Batch shape:", train_inputs.shape)
print("Supervised token count:", supervised_mask.sum().item())


Batch shape: torch.Size([4, 38])
Supervised token count: 26


In [21]:
import copy

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Keep the original pretrained model untouched.
finetuned_gpt = copy.deepcopy(pretrained_gpt).to(device)
finetuned_gpt.eval()

# (B=4, T_max=38): predictor input IDs on the training device
train_inputs_device = train_inputs.to(device)

# (B=4, T_max=38): shifted target IDs and ignored positions
train_labels_device = train_labels.to(device)

# (V=65, C=256): original pretrained output projection
original_lm_head = pretrained_gpt.lm_head.weight.detach().clone()

with torch.no_grad():
    # (B=4, T_max=38, V=65): baseline next-token logits
    initial_logits = finetuned_gpt(train_inputs_device)

    # (): assistant-only NLL before fine-tuning
    initial_sft_loss = F.cross_entropy(
        initial_logits.reshape(-1, pretrained_config.vocab_size),
        train_labels_device.reshape(-1),
        ignore_index=-100,
    )

print("Initial SFT loss:", initial_sft_loss.item())


Initial SFT loss: 3.057330846786499


In [22]:
learning_rate = 2e-4
training_steps = 40
max_grad_norm = 1.0

optimizer = torch.optim.AdamW(
    finetuned_gpt.parameters(), lr=learning_rate, weight_decay=0.0
)

finetuned_gpt.train()

train_loss_history: list[float] = []

for step in range(1, training_steps + 1):
    optimizer.zero_grad(set_to_none=True)

    # (B=4, T_max=38, V=65): current next-token predictions
    logits = finetuned_gpt(train_inputs_device)

    # (): mean NLL across the 26 supervised response targets
    loss = F.cross_entropy(
        logits.reshape(-1, pretrained_config.vocab_size),
        train_labels_device.reshape(-1),
        ignore_index=-100,
    )

    loss.backward()

    # (): global parameter gradient norm before clipping
    gradient_norm = torch.nn.utils.clip_grad_norm_(
        finetuned_gpt.parameters(), max_norm=max_grad_norm
    )

    optimizer.step()

    train_loss_history.append(loss.item())

    if step in (1, 10, 20, 40):
        print(
            f"Step {step:02d} | "
            f"Loss {loss.item():.4f} | "
            f"Grad norm {gradient_norm.item():.4f}"
        )


Step 01 | Loss 3.0573 | Grad norm 18.5646
Step 10 | Loss 0.0844 | Grad norm 0.7866
Step 20 | Loss 0.0007 | Grad norm 0.0161
Step 40 | Loss 0.0002 | Grad norm 0.0028


In [23]:
finetuned_gpt.eval()

with torch.no_grad():
    # (B=4, T_max=38, V=65): logits after supervised updates
    final_logits = finetuned_gpt(train_inputs_device)

    # (): assistant-only NLL after fine-tuning
    final_sft_loss = F.cross_entropy(
        final_logits.reshape(-1, pretrained_config.vocab_size),
        train_labels_device.reshape(-1),
        ignore_index=-100,
    )

# (V=65, C=256): final output projection weights
final_lm_head = finetuned_gpt.lm_head.weight.detach().cpu()

# (): Frobenius norm of the output projection update
weight_change = (final_lm_head - original_lm_head).norm().item()

print("Initial SFT loss:", initial_sft_loss.item())
print("Final SFT loss:", final_sft_loss.item())
print("LM-head weight change:", weight_change)

# Verify that fine-tuning actually modified the copied model.
assert weight_change > 0

# Verify that the original pretrained model remains unchanged.
torch.testing.assert_close(
    pretrained_gpt.lm_head.weight.detach(), original_lm_head
)

assert torch.isfinite(final_sft_loss)

print("Fine-tuning parameter-update checks passed.")


Initial SFT loss: 3.057330846786499
Final SFT loss: 0.00018876561080105603
LM-head weight change: 0.29258882999420166
Fine-tuning parameter-update checks passed.



### First Real SFT Updates — Takeaways

We fine-tuned a copy of our pretrained Tiny GPT using a small
instruction-response dataset.

The model was initialized from pretrained weights rather than
random parameters.

Teacher forcing supplied the original response tokens as context,
while `ignore_index=-100` restricted the loss to assistant response
targets.

The training loop performed backpropagation, gradient clipping,
and AdamW parameter updates.

We inspected the change in the LM-head weights and compared
assistant-only NLL before and after fine-tuning.

A lower training loss demonstrates fitting the supervised examples.
It does not establish general instruction-following or generalization.

The original pretrained model was preserved for later comparison.



### 05.3 Training Diagnostics

A near-zero training loss does not by itself establish
instruction-following generalization.

We compare the pretrained and fine-tuned models using:

- Per-example assistant-only NLL.
- First-response-character probabilities.
- Teacher-forced NLL on one unseen instruction.

These diagnostics distinguish fitting observed responses from
generalizing to unseen instructions.


In [24]:
pretrained_gpt.eval()
finetuned_gpt.eval()

reference_device = next(pretrained_gpt.parameters()).device
tuned_device = next(finetuned_gpt.parameters()).device

with torch.no_grad():
    # (B=4, T=38, V=65): pretrained model logits
    pre_logits = (
        pretrained_gpt(train_inputs.to(reference_device)).detach().cpu()
    )

    # (B=4, T=38, V=65): fine-tuned model logits
    post_logits = finetuned_gpt(train_inputs.to(tuned_device)).detach().cpu()

assert pre_logits.shape == post_logits.shape
print("Logits shape:", pre_logits.shape)


Logits shape: torch.Size([4, 38, 65])


In [25]:
def per_example_sft_nll(
    logits: torch.Tensor,
    labels: torch.Tensor,
) -> torch.Tensor:
    """Compute one assistant-only mean NLL per example.

    Args:
        logits: (B, T, V) vocabulary predictions.
        labels: (B, T) shifted targets with -100 ignored.

    Returns:
        (B,) mean supervised-token NLL for each example.
    """
    batch_size, sequence_length, vocab_size = logits.shape

    # (B, T): selected supervised target positions
    mask = labels.ne(-100)

    # (B,): number of supervised tokens in each example
    counts = mask.sum(dim=1)

    if (counts == 0).any().item():
        raise ValueError("Every example needs supervised targets.")

    # (B, T): unreduced NLL, zero at ignored target positions
    token_nll = F.cross_entropy(
        logits.reshape(-1, vocab_size),
        labels.reshape(-1),
        ignore_index=-100,
        reduction="none",
    ).reshape(batch_size, sequence_length)

    # (B,): mean NLL for each instruction-response example
    return token_nll.sum(dim=1) / counts


# (B=4,): pretrained assistant-only NLL per example
pre_example_nll = per_example_sft_nll(pre_logits, train_labels)

# (B=4,): fine-tuned assistant-only NLL per example
post_example_nll = per_example_sft_nll(post_logits, train_labels)

# (B=4,): supervised target counts
token_counts = train_labels.ne(-100).sum(dim=1)

for index, (instruction, _) in enumerate(training_pairs):
    print(
        f"{instruction:15s} | "
        f"tokens={token_counts[index].item():2d} | "
        f"before={pre_example_nll[index].item():.4f} | "
        f"after={post_example_nll[index].item():.4f}"
    )

# (): reconstruct the token-weighted batch losses
reconstructed_pre_loss = (
    pre_example_nll * token_counts
).sum() / token_counts.sum()

reconstructed_post_loss = (
    post_example_nll * token_counts
).sum() / token_counts.sum()

torch.testing.assert_close(
    reconstructed_pre_loss,
    initial_sft_loss.detach().cpu(),
    rtol=1e-4,
    atol=1e-5,
)

torch.testing.assert_close(
    reconstructed_post_loss, final_sft_loss.detach().cpu(), rtol=1e-4, atol=2e-5
)

print("Per-example and batch-loss consistency passed.")


Say hello       | tokens= 6 | before=3.8168 | after=0.0002
Say farewell    | tokens= 9 | before=2.0548 | after=0.0002
Say thanks      | tokens= 7 | before=2.9810 | after=0.0002
Say yes         | tokens= 4 | before=4.3074 | after=0.0002
Per-example and batch-loss consistency passed.


In [26]:
# (B=4,): predictor indices immediately before each answer
first_positions = torch.tensor(
    [
        len(pretrain_tokenizer.encode(f"User: {instruction}\nAssistant: ")) - 1
        for instruction, _ in training_pairs
    ]
)

# (B=4,): correct first response-character token IDs
first_target_ids = torch.tensor(
    [pretrain_tokenizer.encode(response)[0] for _, response in training_pairs]
)

# (B=4,): example indices
rows = torch.arange(len(training_pairs))

# (B=4, V=65): first-response logits before SFT
pre_first_logits = pre_logits[rows, first_positions]

# (B=4, V=65): first-response logits after SFT
post_first_logits = post_logits[rows, first_positions]

# (B=4,): probability of the correct first character before SFT
pre_first_probs = (
    torch.softmax(pre_first_logits, dim=-1)
    .gather(1, first_target_ids[:, None])
    .squeeze(1)
)

# (B=4,): probability of the correct first character after SFT
post_first_probs = (
    torch.softmax(post_first_logits, dim=-1)
    .gather(1, first_target_ids[:, None])
    .squeeze(1)
)

for index, (instruction, response) in enumerate(training_pairs):
    print(
        f"{instruction:15s} | "
        f"target={response[0]} | "
        f"before={pre_first_probs[index].item():.4f} | "
        f"after={post_first_probs[index].item():.4f}"
    )

assert torch.all((pre_first_probs >= 0) & (pre_first_probs <= 1))
assert torch.all((post_first_probs >= 0) & (post_first_probs <= 1))


Say hello       | target=H | before=0.0009 | after=0.9995
Say farewell    | target=F | before=0.0002 | after=0.9991
Say thanks      | target=T | before=0.0009 | after=0.9997
Say yes         | target=Y | before=0.0005 | after=0.9997


In [27]:
held_out_instruction = "Say welcome"
held_out_response = "Welcome"

held_inputs, held_labels = encode_character_sft_pair(
    held_out_instruction, held_out_response, pretrain_tokenizer
)

# (B=1, T): held-out predictor inputs
held_inputs = held_inputs.unsqueeze(0)

# (B=1, T): held-out shifted labels
held_labels = held_labels.unsqueeze(0)

with torch.no_grad():
    # (B=1, T, V=65): pretrained predictions
    held_pre_logits = (
        pretrained_gpt(held_inputs.to(reference_device)).detach().cpu()
    )

    # (B=1, T, V=65): fine-tuned predictions
    held_post_logits = (
        finetuned_gpt(held_inputs.to(tuned_device)).detach().cpu()
    )

# (): held-out response NLL before SFT
held_pre_nll = per_example_sft_nll(held_pre_logits, held_labels)[0]

# (): held-out response NLL after SFT
held_post_nll = per_example_sft_nll(held_post_logits, held_labels)[0]

print("Held-out instruction:", held_out_instruction)
print("Expected response:", held_out_response)
print("Before SFT:", held_pre_nll.item())
print("After SFT:", held_post_nll.item())

assert torch.isfinite(held_pre_nll)
assert torch.isfinite(held_post_nll)


Held-out instruction: Say welcome
Expected response: Welcome
Before SFT: 2.3555855751037598
After SFT: 3.4502458572387695



### Training Diagnostics — Takeaways

We examined our fine-tuned GPT beyond the aggregate training loss.

Per-example NLL reveals how supervised learning affects individual
instruction-response pairs.

The correct first-response-character probability measures the
model's first autoregressive decision without conditioning on
ground-truth answer characters.

We also evaluated one unseen instruction using teacher-forced NLL.

A decrease in training NLL demonstrates fitting observed examples,
but does not necessarily imply generalization or successful
autoregressive generation.

The next section compares the pretrained and fine-tuned models
through actual response generation.
